# Module 3 — The Full RAG Pipeline

**Duration:** ~105 minutes  
**Goal:** Connect retrieval to Claude and build a complete, production-quality RAG pipeline with prompt engineering.

---

## The Pipeline

```
Question
   │
   ├─ 1. Retrieve  → top-k chunks from ChromaDB
   │
   ├─ 2. Assemble  → system prompt + context + question
   │
   └─ 3. Generate  → Claude produces grounded answer
```

The **prompt template** is the glue. It tells Claude:
- What role to play
- What context it has
- How to handle uncertainty
- What format to use

**Good prompt engineering is 50% of RAG quality.**

In [ ]:
import json
import os
from pathlib import Path

from openai import OpenAI
import chromadb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")

# ── Re-use retrieval setup from Module 2 ───────────────────────────────────
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection = chroma_client.get_or_create_collection(
    name="workshop_rag",
    metadata={"hnsw:space": "cosine"},
)

# Check collection is populated — run Module 2 first if count is 0
print(f"Collection: {collection.count()} documents")
assert collection.count() > 0, "Run 03_vector_store_and_retrieval.ipynb first to index the corpus"

def retrieve(question: str, top_k: int = 5) -> list[dict]:
    q_emb = embed_model.encode(question).tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=top_k,
                                include=["documents", "metadatas", "distances"])
    return [
        {"text": text, "source": meta["source"], "similarity": 1 - dist}
        for text, meta, dist in zip(
            results["documents"][0], results["metadatas"][0], results["distances"][0]
        )
    ]

# ── LLM client ───────────────────────────────────────────────────────────────
client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.ai/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)
FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")
SMART_MODEL = os.getenv("NETLIGHT_MODEL_SMART", "claude-sonnet-4-5")

print(f"LLM: {FAST_MODEL} (fast) / {SMART_MODEL} (smart)")

---
## Part 1 — Prompt Engineering

### The baseline prompt

In [ ]:
SYSTEM_PROMPT_V1 = """\
You are a helpful assistant. Answer questions based on the provided context.
If the context doesn't contain enough information, say so."""

def build_user_prompt_v1(question: str, context_chunks: list[dict]) -> str:
    context = "\n\n---\n\n".join(
        f"[Source: {c['source']}]\n{c['text']}" for c in context_chunks
    )
    return f"""Context:
{context}

Question: {question}"""


# Test with a simple question
question = "What hospitalization costs does DKV cover?"
chunks = retrieve(question, top_k=3)

user_prompt = build_user_prompt_v1(question, chunks)
print("User prompt preview:")
print(user_prompt[:600], "...")

In [ ]:
def generate(system_prompt: str, user_prompt: str, model: str = FAST_MODEL) -> str:
    response = client.chat.completions.create(
            model=model,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
    return response.choices[0].message.content


answer_v1 = generate(SYSTEM_PROMPT_V1, user_prompt)
print(f"Q: {question}\n")
print(f"A (v1):\n{answer_v1}")

### Exercise 3.1 — Improve the system prompt

> **Task:** Write a better `SYSTEM_PROMPT_V2` that:
> 1. Defines a clear persona (e.g., "insurance knowledge assistant")
> 2. Instructs the model to cite sources using the `[Source: X]` tags
> 3. Tells it to explicitly state when context is insufficient
> 4. Requests concise, structured answers
>
> **Hints:**
> - Be specific about the output format
> - Use clear constraints like "only use information from the context below"
> - Instruct Claude on how to handle conflicting information in multiple chunks

In [ ]:
SYSTEM_PROMPT_V2 = """\
# TODO: write an improved system prompt
"""

# Test with the same question
answer_v2 = generate(SYSTEM_PROMPT_V2, build_user_prompt_v1(question, chunks))
print(f"A (v2):\n{answer_v2}")

### Exercise 3.2 — Improve the user prompt template

> **Task:** Write `build_user_prompt_v2` that:
> 1. Shows chunks numbered (e.g., `[1]`, `[2]`) for easy citation
> 2. Includes the similarity score alongside each chunk
> 3. Puts the question clearly at the end with a dedicated label
>
> **Hint:** Test with the question "What happens when an insured person dies?"  
> Does Claude cite the right sources?

In [ ]:
def build_user_prompt_v2(question: str, context_chunks: list[dict]) -> str:
    """
    Build a user prompt with numbered, scored context chunks.
    """
    # TODO: build an improved prompt with numbered chunks and similarity scores
    context_text = ""
    for i, chunk in enumerate(context_chunks, 1):
        # TODO: format each chunk with number, source, similarity, and text
        pass

    return f"""Retrieved Context:
{context_text}

Question: {question}

Answer:"""


# Test
test_question = "What is covered if I need emergency care abroad?"
test_chunks = retrieve(test_question, top_k=4)
prompt_v2 = build_user_prompt_v2(test_question, test_chunks)
print(prompt_v2[:800])

---
## Part 2 — The RAG Class

### Exercise 3.3 — Encapsulate the pipeline

> **Task:** Complete the `RAGPipeline` class below.
>
> The `ask` method should:
> 1. Retrieve relevant chunks
> 2. Build the prompt
> 3. Call Claude
> 4. Return both the answer AND the retrieved chunks (needed for evaluation later!)
>
> **Hints:**
> - Keep the prompt builder and model as configurable attributes
> - Return a `dict` with `answer`, `contexts`, `sources`, `question`

In [ ]:
class RAGPipeline:
    def __init__(
        self,
        collection,
        embed_model,
        llm_client: OpenAI,
        model: str = FAST_MODEL,
        top_k: int = 5,
        system_prompt: str = SYSTEM_PROMPT_V1,
        min_similarity: float = 0.0,
    ):
        self.collection = collection
        self.embed_model = embed_model
        self.llm_client = llm_client
        self.model = model
        self.top_k = top_k
        self.system_prompt = system_prompt
        self.min_similarity = min_similarity

    def retrieve(self, question: str) -> list[dict]:
        # TODO: use self.embed_model and self.collection to retrieve chunks
        # filter by self.min_similarity
        pass

    def build_prompt(self, question: str, chunks: list[dict]) -> str:
        # TODO: build the user prompt (reuse one of your prompt builders)
        pass

    def ask(self, question: str) -> dict:
        """
        Returns:
            dict with keys: question, answer, contexts (list of text), sources (list of str)
        """
        # TODO: retrieve → build prompt → generate → return result dict
        pass


# Instantiate and test
rag = RAGPipeline(
    collection=collection,
    embed_model=embed_model,
    llm_client=client,
    model=FAST_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V1,
)

result = rag.ask("What is covered under DKV Hospi Essential?")
print(f"Q: {result.get('question')}\n")
print(f"A: {result.get('answer')}\n")
print(f"Sources: {result.get('sources')}")

---
## Part 3 — Evaluation Preview

Let's run the full QA test set through our pipeline and inspect results.

In [ ]:
# Load ground truth questions
with open("../data/sample_qa.json") as f:
    qa_data = json.load(f)

# Run first 5 questions
print("Running RAG on test questions...\n")
results = []
for qa in qa_data["questions"][:5]:
    result = rag.ask(qa["question"])
    result["ground_truth"] = qa["ground_truth"]
    results.append(result)
    print(f"Q: {qa['question'][:60]}...")
    print(f"A: {result['answer'][:150]}...\n")

### Exercise 3.4 — Vibe coding experiment

> **Task (open-ended):** Improve the RAG pipeline in one of these ways.  
> Measure the qualitative difference before/after.
>
> Options:
> - **Option A:** Switch to `SMART_MODEL` — does answer quality improve?
> - **Option B:** Increase `top_k` from 3 to 7 — does it help or hurt?
> - **Option C:** Add a follow-up question that asks Claude to summarise its confidence level
> - **Option D:** Try a completely different system prompt persona
>
> **Vibe coding approach:** Describe the change you want in natural language to your AI coding assistant, then evaluate the result.

In [ ]:
# Your experiment here
# ...

# Compare answers side-by-side
comparison_question = "How does DKV Hospi Essential differ from DKV Hospi Premium?"

# rag_v1 = RAGPipeline(..., top_k=3)
# rag_v2 = RAGPipeline(..., top_k=7)
# result_v1 = rag_v1.ask(comparison_question)
# result_v2 = rag_v2.ask(comparison_question)

# print("=== top_k=3 ===")
# print(result_v1["answer"])
# print("\n=== top_k=7 ===")
# print(result_v2["answer"])

---
## Reflection Questions

1. **Faithfulness vs creativity:** The system prompt constrains Claude to only use the context.  
   When would you *want* Claude to go beyond the context?

2. **Token cost:** Each retrieved chunk adds tokens to the prompt. With `top_k=5` at 200 words each, you're adding ~1000 words of context. What's the trade-off?

3. **Multi-hop questions:** "Which is riskier — life insurance or reinsurance?" requires reasoning across two topics. Does your pipeline handle this?

4. **Response latency:** How could you reduce the time-to-first-token for users?
   (Hint: Claude supports streaming responses.)

---

## What we built

- [x] Baseline + improved system prompts
- [x] Numbered, scored context prompt template
- [x] `RAGPipeline` class with configurable parameters
- [x] End-to-end test run on the QA dataset

## Next: Module 4 — Evaluation with RAGAS

We have answers. Now we need to **measure** their quality systematically.  
Open `05_evaluation.ipynb` →